# Triage_HF Statistical & Graphical Analytical Model
* Started date: 02/07/2024 - 05:09 AM
* Data Management Team, Triage_HF

# 1. Dataset cleaning
We will import our dataset and perform a first cleanup to begin to understand which columns and values we are dealing with.

In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import datetime as dt
import plotly.io as pio
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer 
import string

# import gensim
# from gensim.models import Word2Vec
# from gensim.models import KeyedVectors

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

#pio.renderers.default = "browser"
pd.options.mode.chained_assignment = None
pd.set_option('display.max.rows', None)
pd.set_option('display.max.columns', None)

There are two ways to import our data set:
1. Locally: if we have the repository locally, we will use the following line

In [4]:
# Locally method
train_df = pd.read_csv('../dataset/raw/TRIAGE_2024.csv')

2. Remotely: if we want to connect the dataset from github, we must import it using the following line



**WARNING**: every time we want to use this form, we must generate the token again.

In [5]:
# Github method
# train_df = pd.read_csv('https://raw.githubusercontent.com/Adriellevy/Triage_HF/main/Data%20Analysis%20and%20Reports/dataset/raw/TRIAGE%202024.csv?token=GHSAT0AAAAAACLM5VZGUTNJJPTGADJMMUKEZOQ3JLA')

In [6]:
train_df.head(30)

,3+-99999|a,[ñ_MJ,NOMBRE Y APELLIDO,MOTIVO DE CONSULTA,BOX,TRIAGE,ENFERMERO,MEDICO,DESTINO,A,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,FECHA: 01/01/2024 ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,01-01,LUSI,FIEBRE Y TOS,18,IV,ERIKA,RODRIGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,01-01,LO PINTO CARLOS,FIEBRE Y TOS,17,IV,SOLEDAD G,SOLEDAD,alta,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,01-01,BANDERA,TOS,12,IV,ERIKA,RODRIGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,01-01,TOMINO EDUARDO,HTA,12,IV,SOLEDAD G,SOLEDAD,ALTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,01-01,D IORIO ROLANDO EMILIO,FIEBRE,5,IV,ERIKA,SOLEDAD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,01-01,DELGADO HORACIO,SANGRADO DE BOLSILLO MCP,16,III,SOLEDAD G,SOLEDAD,alta,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,01-01,CASTILLO NOEL,ASTENIA,18,IV,ERIKA,RODRIGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,8,01-01,DUBROVSKY ALBERTO LUIS,hematuria hace 3 dias,15,IV,SOLEDAD G,SOLEDAD,ALTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,01-01,INVENENATO RUPERTO DOMINGO,FIEBRE,5,IV,ERIKA,SOLEDAD,NaN,KPC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1.1 Columns rename

At first, you can see how certain columns are wrongly named, so I will assign a corresponding name to them:

In [7]:
cols_rename = { '[ñ_MJ': 'FECHA DE INGRESO',
                'A': 'AISLADO' }
train_df.rename(columns = cols_rename, inplace = True)

In addition, each time you move to the next day according to the date of entry, the first column returns to 1:

In [8]:
train_df.loc[train_df['FECHA DE INGRESO'] == '01-01', '3+-99999|a'].iloc[0] == train_df.loc[train_df['FECHA DE INGRESO'] == '01-02', '3+-99999|a'].iloc[0]

True

Therefore, I will name that first column "NUMERO DE TURNO" in reference to the fact that the turns are reset at the beginning of the next day:

In [9]:
col_rename = { '3+-99999|a': 'NUMERO DE TURNO' }
train_df.rename(columns = col_rename, inplace = True)

## 1.2 Removing headers and null rows

We will remove the header that appears every time a new day begins:

In [10]:
train_df = train_df[train_df['NUMERO DE TURNO'].str.contains('FECHA|N°') == False]

In addition, we will remove the last rows of the dataset that do not contain any value in first and last name

In [11]:
train_df = train_df[train_df['NOMBRE Y APELLIDO'].notnull()]

## 1.3 Changing Triage Level data type

For possible machine learning models in the future, the triage level should be int dtype:

In [12]:
train_df['TRIAGE'].dtype

dtype('O')

In [13]:
train_df['TRIAGE'] = train_df['TRIAGE'].str.strip().str.upper()

In [14]:
vals_rename = { 'I': 1, 'II': 2, 'III': 3, 'IV': 4 }
train_df['TRIAGE'] = train_df['TRIAGE'].replace(vals_rename)

Those rows not containing 1, 2, 3, or 4 will be taken as null (using errors coerce)

In [15]:
train_df['TRIAGE'] = pd.to_numeric(train_df['TRIAGE'], downcast = "signed", errors = 'coerce')

In [16]:
train_df['TRIAGE'].dtype

dtype('float64')

## 1.4 Changing Entry Date data type

We will change the object type of the entry date to date type. This makes date manipulation much easier and more readable:

In [17]:
train_df['FECHA DE INGRESO'].dtype

dtype('O')

In [18]:
train_df['FECHA DE INGRESO'] = train_df['FECHA DE INGRESO'] + '-2024'

In [19]:
train_df['FECHA DE INGRESO'] = pd.to_datetime(train_df['FECHA DE INGRESO'], format = '%d-%m-%Y', errors = 'coerce')

In [20]:
train_df['FECHA DE INGRESO'].dtype

dtype('<M8[ns]')

## 1.5 Changing Isolated data type

In [21]:
train_df['AISLADO'].unique()

array([nan, 'KPC', 'NO', 'SI', 'No', '724', 'Si', 'ECOLI METALO'],
      dtype=object)

In [22]:
data = {'AISLADO': [np.nan, 'KPC', 'NO', 'SI', 'No', '724', 'Si', 'A', 'ECOLI METALO']}
mapping = {np.nan: False, 'NO': False, 'No': False,
           'KPC': True, 'ECOLI METALO': True, 'SI': True, 'Si': True, 'A': True}
train_df['AISLADO'] = train_df['AISLADO'].map(mapping)

In [23]:
train_df = train_df[train_df['AISLADO'] != '724']

In [24]:
train_df['AISLADO'] = train_df['AISLADO'].astype(bool)

## 1.6 Generating ALTA column

In [25]:
train_df['DESTINO'].unique()

array([nan, 'alta', 'ALTA', '624', 'ALTA ', 'FUGA', '506', '?', 'INT',
       '715', '721', '512', '717', '820', '622', 'ALT.VOL', '312', '812',
       'HMD/ 315', 'HMD', '316', '301', '806', 'Alta', '802', '809',
       'PISO', '315', '722', '805', 'alta ', 'OBITO', 'Traslado', ' ',
       '    ', '311', '808', 'ALTA VOLUNT', '302', '813', '818', '304',
       'ALTA V ', '814', '613', '619', '602', '615', 'ALTA MEDICA', '314',
       '626', '607', '726', 'ALTA VOL', '926', '922', '920', '917', '916',
       '711', 'DERIVACION', '714', '719', '  ', '724', 'FUGA ', 'DERIV',
       'AL. VOL', '915', '918', 'QUIROFANO', 'ALTA VOLUNT.', '310',
       'TRASLADO ', '928', '919', '616', 'QUIROFANO ', '628', '909',
       '804', 'ALTA VOLUNTARIA', 'DERIVACION ', '720', '803', 'DERIVAC',
       'alTA', '908', '709', '621', '306',
       'ALTA                                                                                                                                                           

In [26]:
train_df = train_df[~train_df['DESTINO'].isin(['?', 'INT', 'HMD/ 315', 'HMD', ' ', '    ', '  ', ])]

In [27]:
train_df['ALTA'] = train_df['DESTINO'].str.contains(
    'alta|obito|traslado|derivacion|AL. VOL|DERIVAC',
    case=False,  
    regex=True   
)
train_df['ALTA'].fillna(True, inplace=True)

Finally, empty and unnamed columns can be visualized. We will remove them:

In [28]:
cols_to_keep = ['NUMERO DE TURNO', 'FECHA DE INGRESO', 'NOMBRE Y APELLIDO', 'MOTIVO DE CONSULTA', 'BOX', 'TRIAGE', 'ENFERMERO', 'MEDICO', 'DESTINO', 'ALTA', 'AISLADO']
train_df = train_df[cols_to_keep]

This is how our dataframe would look at first:

In [29]:
train_df.head()

,NUMERO DE TURNO,FECHA DE INGRESO,NOMBRE Y APELLIDO,MOTIVO DE CONSULTA,BOX,TRIAGE,ENFERMERO,MEDICO,DESTINO,ALTA,AISLADO
1,1,2024-01-01,LUSI,FIEBRE Y TOS,18,4.0,ERIKA,RODRIGO,NaN,True,False
2,2,2024-01-01,LO PINTO CARLOS,FIEBRE Y TOS,17,4.0,SOLEDAD G,SOLEDAD,alta,True,False
3,3,2024-01-01,BANDERA,TOS,12,4.0,ERIKA,RODRIGO,NaN,True,False
4,4,2024-01-01,TOMINO EDUARDO,HTA,12,4.0,SOLEDAD G,SOLEDAD,ALTA,True,False
5,5,2024-01-01,D IORIO ROLANDO EMILIO,FIEBRE,5,4.0,ERIKA,SOLEDAD,NaN,True,False


In [30]:
def grafico_barras(df, p_x, p_y, title, x_title, y_title, media = None):
    fig = px.bar(df, x=p_x, y=p_y, barmode="group")
    fig.update_layout(title=title, xaxis_title=x_title, yaxis_title=y_title, title_x=0.5)
    fig.update_traces(texttemplate='%{y}', textposition='outside')
    if(media):
        media = df[p_y].mean()
        fig.add_trace(go.Scatter(x=df[p_x],
                                 y=[media] * len(df),
                                 mode='lines',
                                 name='Media',
                                 line=dict(color='red', width=2, dash='dash'),
                                 hovertemplate='%{y:.2f}'))
        fig.add_annotation(
        xref='paper', yref='y',
        x=-0.03, y=media,
        text=f'{media:.2f}',
        showarrow=False,
        font=dict(color='red'))
    fig.show()


In [31]:
def filtros(df, desde = None, hasta = None, nombre_y_apellido = None, motivo_de_consulta = None, box = None, triage = None, medico = None, enfermero = None, alta = None, aislado = None):
    if(desde != None and hasta != None):
        desde = pd.to_datetime(desde, format='%d-%m-%Y', errors='coerce') 
        hasta = pd.to_datetime(hasta, format='%d-%m-%Y', errors='coerce')   
        df = df[(df['FECHA DE INGRESO'] >= desde) & (df['FECHA DE INGRESO'] <= hasta)]
        
    df['FECHA DE INGRESO'] = df['FECHA DE INGRESO'].dt.strftime('%d-%m-%Y')      
       
    if(nombre_y_apellido != None):
        df = df[df['NOMBRE Y APELLIDO'] == nombre_y_apellido]

    if(motivo_de_consulta != None):
        df = df[df['MOTIVO DE CONSULTA'] == motivo_de_consulta]

    if(box != None):
        df = df[df['BOX'] == box]

    if(triage != None):
        df = df[df['TRIAGE'] == triage]

    if(medico != None):
        df = df[df['MEDICO'] == medico]

    if(enfermero != None):
        df = df[df['ENFERMERO'] == enfermero]

    if(alta != None):
        df = df[df['ALTA'] == alta]

    if(aislado != None):
        df = df[df['AISLADO'] == aislado]

    return df

In [32]:
def cant_pacientes_fecha(desde = None, hasta = None, nombre_y_apellido = None, motivo_de_consulta = None, box = None, triage = None, medico = None, enfermero = None, alta = None, aislado = None):
    train_df_acotado = train_df.groupby('FECHA DE INGRESO', sort=False).size().reset_index()
    train_df_acotado.rename(columns={0: 'CANTIDAD DE PACIENTES'}, inplace=True)
    train_df_acotado = filtros(train_df_acotado, desde, hasta, nombre_y_apellido, motivo_de_consulta, box, triage, medico, enfermero, alta, aislado)
    return train_df_acotado


In [33]:
df = cant_pacientes_fecha(desde = None, hasta = None,
             nombre_y_apellido = None, motivo_de_consulta = None,
             box = None, triage = None,
             medico = None, enfermero = None,
             alta = None, aislado = None)

grafico_barras(df = df, 
               p_x='FECHA DE INGRESO', p_y='CANTIDAD DE PACIENTES', 
               x_title='Cantidad de Pacientes por Fecha de Ingreso', y_title='Fecha de Ingreso', title='Cantidad de Pacientes', 
               media=True)


In [34]:
def top_consultas_fecha(desde=None, hasta=None, top = 10, order=None, nombre_y_apellido = None, motivo_de_consulta = None, box = None, triage = None, medico = None, enfermero = None, alta = None, aislado = None):
    top_motivos = train_df.copy(deep=True)
    top_motivos = filtros(top_motivos, desde, hasta, nombre_y_apellido, motivo_de_consulta, box, triage, medico, enfermero, alta, aislado)
    
    motivos_count = top_motivos['MOTIVO DE CONSULTA'].value_counts()

    top_motivos = motivos_count.nlargest(top).reset_index()
    top_motivos.columns = ['MOTIVO DE CONSULTA', 'CANTIDAD DE CONSULTAS']

    if order == 'asc':
        top_motivos.sort_values(by='CANTIDAD DE CONSULTAS', ascending = True, inplace = True)
        
    return top_motivos

In [35]:
df = top_consultas_fecha(desde = None, hasta=None,
             top = 15, order = 'asc',
             nombre_y_apellido = None, motivo_de_consulta = None,
             box = None, triage = None,
             medico = None, enfermero = None,
             alta = None, aislado = None)

grafico_barras(df = df, 
               p_x='MOTIVO DE CONSULTA', p_y='CANTIDAD DE CONSULTAS', 
               title='Cantidad de Consultas', x_title='Motivos de Consulta mas Frecuentes', y_title='Motivo de Consulta', 
               media=False)

In [36]:
def cant_pacientes_triage(desde = None, hasta = None, nombre_y_apellido = None, motivo_de_consulta = None, box = None, triage = None, medico = None, enfermero = None, alta = None, aislado = None):
    train_df_acotado = train_df.groupby('TRIAGE', sort=False).size().reset_index()
    train_df_acotado.rename(columns={0: 'CANTIDAD DE PACIENTES'}, inplace=True)
    train_df_acotado = filtros(train_df_acotado, desde, hasta, nombre_y_apellido, motivo_de_consulta, box, triage, medico, enfermero, alta, aislado)
    return train_df_acotado


In [37]:
# df = cant_pacientes_triage(desde = None, hasta = None,
#              nombre_y_apellido = None, motivo_de_consulta = None,
#              box = None, triage = None,
#              medico = None, enfermero = None,
#              alta = None, aislado = None)

# grafico_barras(df = df, 
#                p_x='TRIAGE', p_y='CANTIDAD DE PACIENTES', 
#                x_title='Nivel de Triage', y_title='Cantidad de pacientes', title='Cantidad de Pacientes por Nivel de Triage', 
#                media=True)

In [38]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Adri\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Adri\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Adri\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [39]:
def preprocess_text(text):
    # convert to lowercase
    text = str(text).lower()

    # remove punctuation
    text = ''.join([char for char in text if char not in string.punctuation])

    # separate by syllables
    tokens = nltk.word_tokenize(text, 'spanish')

    # eliminate common words without meaning
    stop_words = set(stopwords.words('spanish'))
    filtered_tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization (converting words to their base form)
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]
    return ' '.join(lemmatized_tokens)

In [40]:
train_df['MOTIVO DE CONSULTA'] = train_df['MOTIVO DE CONSULTA'].apply(preprocess_text)

display(train_df['MOTIVO DE CONSULTA'].head(20))

1                       fiebre tos
2                       fiebre tos
3                              tos
4                              hta
5                           fiebre
6            sangrado bolsillo mcp
7                          astenia
8             hematuria hace 3 dia
9                           fiebre
10               dolor rodilla izq
11                    dolor lumbar
12    dolor hueco popliteo derecho
13                         diarrea
14                      odinofagia
15                         sincope
16                         sincope
17                       forunculo
18                dolor costal izq
20                   palpitaciones
21                       hematuria
Name: MOTIVO DE CONSULTA, dtype: object

In [41]:
#------------------------------------------------------------TODA ESTA CELDA FUE HECHA CON EL OBJETIVO DE AGRUPAR SINTOMAS -------------------------------------------------------

# sintomas_rellenados_por_enfermeros = [sentence.split() for sentence in train_df['MOTIVO DE CONSULTA']]


# # Cargar el modelo preentrenado
# model = KeyedVectors.load_word2vec_format('..\dataset\Pre-Trained-Word2Vec-Files\GoogleNews-vectors-negative300.bin', binary=True)

# # Ahora puedes usar el modelo para encontrar los síntomas más similares
# sintomas_no_entrenados=[]
# for lista_sintomas in sintomas_rellenados_por_enfermeros:
#     print(f"Lista de síntomas: {lista_sintomas}")
#     # Para cada síntoma en la lista...
#     for sintoma in lista_sintomas:
#         print(f"  Síntoma: {sintoma}")
#         # Verificar si el síntoma está en el vocabulario del modelo
#         if sintoma in model.key_to_index:
#             # Encontrar el síntoma más similar
#             similares = model.most_similar(sintoma, topn=1)
#             for sim in similares:
#                 print(f"    Síntoma más similar: {sim[0]} (similitud: {sim[1]})")
#         else:
#             sintomas_no_entrenados.append(sintoma)
#             #print(f"    El síntoma '{sintoma}' no está en el vocabulario del modelo.")

#------------------------------------------------------ REMPLAZAR WORD2VEC POR FASTTEXT ------------------------------------------
# sintomas_rellenados_por_enfermeros = [sentence.split() for sentence in train_df['MOTIVO DE CONSULTA']]
# #print(sintomas_rellenados_por_enfermeros)
# #Entreno el modelo usando los sintomas ideales (esto puede variari no solo por cada centro sino por temporada)
# #Agegue a la lista que nos pasaron el sintoma tos, habría que evaluar otros posibles casos de eso
# sintomas_ideales=[['convulsiones'],
#     ['trauma de cráneo'],
#     ['dolor torácico'],
#     ['dorsal'],
#     ['dolor abdominal'],
#     ['lumbar'],
#     ['cefalea'],
#     ['déficit motor'],
#     ['disartria'],
#     ['afasia'],
#     ['pérdida aguda de visión'],
#     ['disnea'],
#     ['otro dolor en curso'],
#     ['sobredosis de fármacos'],
#     ['ingesta de tóxicos'],
#     ['sangrado digestivo'],
#     ['fiebre'],
#     ['tos']]
# #corpus = sintomas_ideales + sintomas_rellenados_por_enfermeros
# model = KeyedVectors.load('..\dataset\Pre-Trained-Word2Vec-Files\complete.kv', mmap='r')

# # SintomaIngresado = sintomas_rellenados_por_enfermeros[0][0]
# # SintomaIngresado = "fiebre "
# # if SintomaIngresado in model.wv.key_to_index:
# # for sintoma_ideal in sintomas_ideales:
# #     sintoma_ideal = sintoma_ideal[0]
# #     if sintoma_ideal in model.wv.key_to_index:
# #         similitud = model.wv.similarity(SintomaIngresado, sintoma_ideal)
# #     if similitud > mayor_similitud:
# #         mayor_similitud = similitud
# #         sintoma_mas_similar = sintoma_ideal
# # print(f"El síntoma ideal más similar a '{SintomaIngresado}' es '{sintoma_mas_similar}' con una similitud de {mayor_similitud}.")
# # else:
# #    print("El síntoma ingresado no está en el vocabulario del modelo.")


# # sintomas_no_entrenados=[]
# # for lista_sintomas in sintomas_rellenados_por_enfermeros:
# #     print(f"Lista de síntomas: {lista_sintomas}")
# #     Para cada síntoma en la lista...
# #     for sintoma in lista_sintomas:
# #         print(f"  Síntoma: {sintoma}")
# #         Verificar si el síntoma está en el vocabulario del modelo
# #         if sintoma in model.wv.key_to_index:
# #             Encontrar el síntoma más similar
# #             similares = model.wv.most_similar(sintoma, topn=1)
# #             for sim in similares:
# #                 print(f"    Síntoma más similar: {sim[0]} (similitud: {sim[1]})")
# #         else:
# #             sintomas_no_entrenados.append(sintoma)
# #             print(f"    El síntoma '{sintoma}' no está en el vocabulario del modelo.")


# sintoma_mas_similar = 'err'
# mayor_similitud = -1
# sintoma_ingresado = 'cefalea'  # Aquí va el síntoma ingresado por el usuario

# for sintoma_ideal in sintomas_ideales:
#     try:
#         similitud = model.similarity(sintoma_ingresado, sintoma_ideal[0])
#         if similitud > mayor_similitud:
#             mayor_similitud = similitud
#             sintoma_mas_similar = sintoma_ideal[0]
#     except KeyError:
#         continue

# print(f'El síntoma más similar a "{sintoma_ingresado}" es "{sintoma_mas_similar}" con una similitud de {mayor_similitud}.')



# #Le asigno un valor a cada palabra
# for sintoma in sintomas_rellenados_por_enfermeros:
#     print(f"Síntoma: {sintoma}")
#     similares = model.wv.most_similar(sintoma)
#     for sim in similares:
#         print(f"  - {sim[0]}: {sim[1]}")


# sintomas_rellenados_por_enfermeros = train_df['MOTIVO DE CONSULTA'].tolist()
# #print(sintomas_rellenados_por_enfermeros)
# #Entreno el modelo usando los sintomas ideales (esto puede variari no solo por cada centro sino por temporada)
# #Agegue a la lista que nos pasaron el sintoma tos, habría que evaluar otros posibles casos de eso
# sintomas_ideales=[['convulsiones'],
#     ['trauma de cráneo'],
#     ['dolor torácico'], 
#     ['dorsal'],
#     ['dolor abdominal'], 
#     ['lumbar'],
#     ['cefalea'],
#     ['déficit motor'], 
#     ['disartria'], 
#     ['afasia'],
#     ['pérdida aguda de visión'],
#     ['disnea'],
#     ['otro dolor en curso'],
#     ['sobredosis de fármacos'],
#     ['ingesta de tóxicos'],
#     ['sangrado digestivo'],
#     ['fiebre'],
#     ['tos']]
# model = Word2Vec(sintomas_ideales, vector_size=50, window=5, min_count=1, sg=0)
# # #Le asigno un valor a cada palabra
# # for sintoma in sintomas_rellenados_por_enfermeros:
# #     print(f"Síntoma: {sintoma}")
# #     similares = model.wv.most_similar(sintoma)
# #     for sim in similares:
# #         print(f"  - {sim[0]}: {sim[1]}")
#-------------------------------------------------- SECCION DE CODIGO AUXILIAR ---------------------------------------------
# sintomas_no_relacionados=[]
# for sintoma in sintomas_rellenados_por_enfermeros:
#     # Para cada síntoma en la lista...
#     print(f"  Síntoma: {sintoma}")
#     # Verificar si el síntoma está en el vocabulario del modelo
#     if sintoma in model.wv.key_to_index:
#     # Encontrar el síntoma más similar
#         similares = model.wv.most_similar(sintoma, topn=1)
#         for sim in similares:
#             print(f"    Síntoma más similar: {sim[0]} (similitud: {sim[1]})")
#     else:
#         sintomas_no_relacionados.append(sintoma)
#             #print(f"    El síntoma '{sintoma}' no está en el vocabulario del modelo.")

In [42]:
print("Cantidad de pacientes: " + str(train_df.shape[0]))
df = top_consultas_fecha(desde = "1-1-2024", hasta="3-3-2024",
             top = 50, order = 'asc',
             nombre_y_apellido = None, motivo_de_consulta = None,
             box = None, triage = None,
             medico = None, enfermero = None,
             alta = None, aislado = None)

grafico_barras(df = df, 
               p_x='MOTIVO DE CONSULTA', p_y='CANTIDAD DE CONSULTAS', 
               title='Cantidad de Consultas', x_title='Motivos de Consulta mas Frecuentes', y_title='Motivo de Consulta', 
               media=False)

Cantidad de pacientes: 1425


In [43]:

sintomas_rellenados_por_enfermeros = train_df['MOTIVO DE CONSULTA'].tolist()
resultados = []
# Lista de síntomas ideales
sintomas_ideales = ['convulsiones', 'trauma de cráneo', 'dolor torácico', 'dorsal', 'dolor abdominal', 'lumbar', 'cefalea', 'déficit motor', 'disartria', 'afasia', 'pérdida aguda de visión', 'disnea', 'otro dolor en curso', 'sobredosis de fármacos', 'ingesta de tóxicos', 'sangrado digestivo', 'fiebre', 'tos','dt','hta','diarrea','palpitaciones']
print(len(sintomas_ideales)) 
# Diccionario para los síntomas sin similitud
sintomas_sin_similitud = {}
#DataFrame para guardar los resultados
df_relacion_pacientes_sintomas = pd.DataFrame(columns=['Paciente', 'Sintoma', 'Similitud'])
# Iteramos sobre cada paciente
for index, row in train_df.iterrows():
    # Síntoma del paciente
    sintoma_paciente = [row['MOTIVO DE CONSULTA']]

    # Unimos las listas
    textos = sintoma_paciente + sintomas_ideales

    # Calculamos TF-IDF
    vectorizer = TfidfVectorizer().fit_transform(textos)

    # Obtenemos la matriz de similitud del coseno
    similitud_coseno = cosine_similarity(vectorizer)
    indices_ordenados = np.argsort(similitud_coseno[0][1:])[::-1]

    if similitud_coseno[0][indices_ordenados[0]+1] == 0:
        if sintoma_paciente[0] in sintomas_sin_similitud:
            sintomas_sin_similitud[sintoma_paciente[0]] += 1
        else:
            sintomas_sin_similitud[sintoma_paciente[0]] = 1
    else:
        # Guardamos el paciente, los tres síntomas con mayor similitud y sus valores de similitud
        resultados.append({
            'Paciente': index,
            'Triage':train_df['TRIAGE'][index],
            'Sintoma real': sintoma_paciente,  
            'Sintoma1': sintomas_ideales[indices_ordenados[0]], 
            'Similitud1': similitud_coseno[0][indices_ordenados[0]+1],
            'Sintoma2': sintomas_ideales[indices_ordenados[1]], 
            'Similitud2': similitud_coseno[0][indices_ordenados[1]+1],
            'Sintoma3': sintomas_ideales[indices_ordenados[2]], 
            'Similitud3': similitud_coseno[0][indices_ordenados[2]+1],
            'Box':train_df['BOX'][index]
        })

df_relacion_pacientes_sintomas = pd.DataFrame(resultados)
display(df_relacion_pacientes_sintomas.head(5))
#Aca ordeno el diccionario de sintomas que no tuvieron alguna prediccion.
#Esta lista nos va ayudar a ir mejorando el sistema de predicciones ya que 
#nos va a permitir tener un control exaustivo sobre que tan util esta siendo  
#nuestro sistema de prediccion.
sintomas_sin_similitud_ordenados = {k: v for k, v in sorted(sintomas_sin_similitud.items(), key=lambda item: item[1], reverse=True)}

#Cantidad de sintomas que no se pudieron predecir: 
cantidad_de_sintmas=sum(sintomas_sin_similitud.values())
display("Cantidad de sintomas sin alguna similitud exitosa: "+str(cantidad_de_sintmas) )
#display(sintomas_sin_similitud_ordenados)
#print(sintomas_sin_similitud)
# Ahora puedes agregar la columna de edad a tu DataFrame
# df_sintomas['Edad'] = train_df['Edad']


22


,Paciente,Triage,Sintoma real,Sintoma1,Similitud1,Sintoma2,Similitud2,Sintoma3,Similitud3,Box
0,1,4.0,[fiebre tos],tos,0.707107,fiebre,0.707107,palpitaciones,0.0,18
1,2,4.0,[fiebre tos],tos,0.707107,fiebre,0.707107,palpitaciones,0.0,17
2,3,4.0,[tos],tos,1.000000,palpitaciones,0.000000,afasia,0.0,12
3,4,4.0,[hta],hta,1.000000,palpitaciones,0.000000,afasia,0.0,12
4,5,4.0,[fiebre],fiebre,1.000000,palpitaciones,0.000000,afasia,0.0,5


'Cantidad de sintomas sin alguna similitud exitosa: 701'

## K-MEANS (Aprendizaje no supervisado)

In [44]:
#------------------------------------- prediccion de Triage KMEANs ------------------------------

# Supongamos que tus datos están en un DataFrame de pandas llamado df
# Convertimos los síntomas a números
df_relacion_pacientes_sintomas_kmeans = df_relacion_pacientes_sintomas.copy()

le = LabelEncoder()
df_relacion_pacientes_sintomas_kmeans['Sintoma1'] = le.fit_transform(df_relacion_pacientes_sintomas_kmeans['Sintoma1'])
df_relacion_pacientes_sintomas_kmeans['Sintoma2'] = le.fit_transform(df_relacion_pacientes_sintomas_kmeans['Sintoma2'])
df_relacion_pacientes_sintomas_kmeans['Sintoma3'] = le.fit_transform(df_relacion_pacientes_sintomas_kmeans['Sintoma3'])

# Creamos la matriz de características
X = df_relacion_pacientes_sintomas_kmeans[['Sintoma1', 'Similitud1', 'Sintoma2', 'Similitud2', 'Sintoma3', 'Similitud3']].values

# Crear y entrenar el modelo KMeans
kmeans = KMeans(n_clusters=4)  # Asume que hay 4 niveles de triage
kmeans.fit(X)

# Paciente de Ejemplo
nuevo_paciente  = {
    'Sintoma1': ['tos'],
    'Similitud1': [0.707107],
    'Sintoma2': ['fiebre'],
    'Similitud2': [0.707107],
    'Sintoma3': ['palpitaciones'],
    'Similitud3': [0.000000]
}
df_nuevo_paciente = pd.DataFrame(nuevo_paciente)

df_nuevo_paciente['Sintoma1'] = le.fit_transform(df_nuevo_paciente['Sintoma1'])
df_nuevo_paciente['Sintoma2'] = le.fit_transform(df_nuevo_paciente['Sintoma2'])
df_nuevo_paciente['Sintoma3'] = le.fit_transform(df_nuevo_paciente['Sintoma3'])
nuevo_paciente_lista = df_nuevo_paciente.values.tolist()
nuevo_paciente = [[1, 0.8, 2, 0.7, 3, 0.6]]  # Debe ser una lista de listas
print(nuevo_paciente_lista,nuevo_paciente)
# Ahora puedes predecir el nivel de triage para nuevos pacientes
# Supongamos que tienes un nuevo paciente con los siguientes síntomas y valores:
prediccion = kmeans.predict(nuevo_paciente_lista)

print(f'El nivel de triage más probable para este paciente es: {prediccion[0]}')

[[0.0, 0.707107, 0.0, 0.707107, 0.0, 0.0]] [[1, 0.8, 2, 0.7, 3, 0.6]]
El nivel de triage más probable para este paciente es: 2


C:\Users\Adri\AppData\Roaming\Python\Python310\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning:

Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.

  File "C:\Users\Adri\AppData\Roaming\Python\Python310\site-packages\joblib\externals\loky\backend\context.py", line 282, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


## ARBOL DE DECISION

#### Visualizacion Matriz + tipo de dato

In [45]:
df_relacion_pacientes_sintomas_ADD_1 = df_relacion_pacientes_sintomas.copy()
df_relacion_pacientes_sintomas_ADD_1.head()

,Paciente,Triage,Sintoma real,Sintoma1,Similitud1,Sintoma2,Similitud2,Sintoma3,Similitud3,Box
0,1,4.0,[fiebre tos],tos,0.707107,fiebre,0.707107,palpitaciones,0.0,18
1,2,4.0,[fiebre tos],tos,0.707107,fiebre,0.707107,palpitaciones,0.0,17
2,3,4.0,[tos],tos,1.000000,palpitaciones,0.000000,afasia,0.0,12
3,4,4.0,[hta],hta,1.000000,palpitaciones,0.000000,afasia,0.0,12
4,5,4.0,[fiebre],fiebre,1.000000,palpitaciones,0.000000,afasia,0.0,5


In [46]:
df_relacion_pacientes_sintomas_ADD_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 724 entries, 0 to 723
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Paciente      724 non-null    int64  
 1   Triage        648 non-null    float64
 2   Sintoma real  724 non-null    object 
 3   Sintoma1      724 non-null    object 
 4   Similitud1    724 non-null    float64
 5   Sintoma2      724 non-null    object 
 6   Similitud2    724 non-null    float64
 7   Sintoma3      724 non-null    object 
 8   Similitud3    724 non-null    float64
 9   Box           713 non-null    object 
dtypes: float64(4), int64(1), object(5)
memory usage: 56.7+ KB


In [47]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn import tree

## Mala Prediccion del Triage con sintoma + box

In [48]:
#En el tutorial de Arboles de decisión con Python es Y
Variable_a_predecir=df_relacion_pacientes_sintomas_ADD_1['Triage']
# Crear el imputador
Variable_a_predecir.fillna(0, inplace=True)

#En el tutorial de Arboles de decisión con Python es X
Variable_predictoras=df_relacion_pacientes_sintomas_ADD_1.iloc[:,3:]
Variable_predictoras.head()

,Sintoma1,Similitud1,Sintoma2,Similitud2,Sintoma3,Similitud3,Box
0,tos,0.707107,fiebre,0.707107,palpitaciones,0.0,18
1,tos,0.707107,fiebre,0.707107,palpitaciones,0.0,17
2,tos,1.000000,palpitaciones,0.000000,afasia,0.0,12
3,hta,1.000000,palpitaciones,0.000000,afasia,0.0,12
4,fiebre,1.000000,palpitaciones,0.000000,afasia,0.0,5


### Codifico los sintomas

In [49]:
le = LabelEncoder()
Variable_predictoras['Sintoma1'] = le.fit_transform(df_relacion_pacientes_sintomas_ADD_1['Sintoma1'])
Variable_predictoras['Sintoma2'] = le.fit_transform(df_relacion_pacientes_sintomas_ADD_1['Sintoma2'])
Variable_predictoras['Sintoma3'] = le.fit_transform(df_relacion_pacientes_sintomas_ADD_1['Sintoma3'])
Variable_predictoras['Box'] = le.fit_transform(df_relacion_pacientes_sintomas_ADD_1['Box'])
#Variable_predictoras.head(70)
if (Variable_predictoras == 'C2').any().any():
    print("'c1' is in the DataFrame.")
else:
    print("'c1' is not in the DataFrame.")

'c1' is not in the DataFrame.


In [50]:
Variable_predictoras.head()

,Sintoma1,Similitud1,Sintoma2,Similitud2,Sintoma3,Similitud3,Box
0,15,0.707107,6,0.707107,5,0.0,11
1,15,0.707107,6,0.707107,5,0.0,10
2,15,1.000000,8,0.000000,0,0.0,5
3,11,1.000000,8,0.000000,0,0.0,5
4,10,1.000000,8,0.000000,0,0.0,15


### Separacion set de datos de entrenamiento y verifcacion

In [51]:
x_train,x_test,y_train,y_test = train_test_split(Variable_predictoras,Variable_a_predecir,train_size=0.75,random_state=0)
x_train.info()
y_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 543 entries, 77 to 684
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Sintoma1    543 non-null    int32  
 1   Similitud1  543 non-null    float64
 2   Sintoma2    543 non-null    int32  
 3   Similitud2  543 non-null    float64
 4   Sintoma3    543 non-null    int32  
 5   Similitud3  543 non-null    float64
 6   Box         543 non-null    int32  
dtypes: float64(3), int32(4)
memory usage: 25.5 KB
<class 'pandas.core.series.Series'>
Index: 543 entries, 77 to 684
Series name: Triage
Non-Null Count  Dtype  
--------------  -----  
543 non-null    float64
dtypes: float64(1)
memory usage: 8.5 KB


##### VERIFICAR QUE NINGUN ELEMENTO SEA TIPO SUBLISTA

In [52]:
for i in range(x_train.shape[1]):
    if isinstance(x_train.iloc[:, i].values[0], list):
        print(f"La característica {i} es una lista.")

In [53]:
#Creamos el modelo
arbol = DecisionTreeClassifier()

#Entrenamos el modelo
arbol_triage=arbol.fit(x_train,y_train)


In [54]:
# fig=plt.figure(figsize=(25,20))
# tree.plot_tree(arbol_triage,feature_names=list(Variable_predictoras.columns.values),class_names=list(Variable_a_predecir.values),filled=True)
# plt.show()

In [55]:
Y_pred=arbol_triage.predict(x_test)
Y_pred

array([2., 3., 2., 3., 2., 3., 2., 0., 0., 4., 4., 3., 4., 0., 3., 3., 3.,
       4., 3., 3., 4., 4., 4., 3., 4., 3., 2., 3., 3., 4., 3., 4., 3., 4.,
       4., 3., 4., 2., 2., 3., 2., 4., 3., 1., 3., 3., 0., 2., 3., 2., 3.,
       3., 4., 3., 3., 3., 3., 2., 4., 2., 2., 3., 2., 3., 2., 2., 3., 2.,
       4., 3., 4., 0., 3., 3., 2., 3., 3., 3., 4., 4., 3., 3., 3., 3., 4.,
       4., 3., 3., 3., 2., 4., 4., 4., 3., 4., 3., 3., 4., 3., 2., 3., 3.,
       2., 3., 3., 4., 4., 3., 0., 2., 3., 0., 4., 3., 4., 4., 3., 3., 0.,
       4., 0., 4., 2., 3., 3., 4., 3., 2., 4., 3., 3., 2., 3., 2., 3., 0.,
       3., 4., 3., 0., 3., 3., 2., 2., 3., 3., 4., 3., 2., 3., 4., 3., 0.,
       3., 3., 3., 2., 4., 3., 4., 3., 0., 3., 4., 3., 3., 3., 2., 3., 3.,
       3., 3., 4., 3., 3., 3., 3., 2., 3., 3., 2.])

#### Matriz de Confusion

In [56]:
matriz_de_verificacion = confusion_matrix(y_test,Y_pred)
matriz_de_verificacion

array([[ 3,  0,  3,  6,  4],
       [ 0,  0,  1,  2,  1],
       [ 1,  0, 10, 12,  5],
       [ 4,  1,  8, 32, 17],
       [ 5,  0, 11, 39, 16]], dtype=int64)

In [57]:
le = LabelEncoder()
Variable_predictoras['Sintoma1'] = le.fit_transform(df_relacion_pacientes_sintomas_ADD_1['Sintoma1'])
Variable_predictoras['Sintoma2'] = le.fit_transform(df_relacion_pacientes_sintomas_ADD_1['Sintoma2'])
Variable_predictoras['Sintoma3'] = le.fit_transform(df_relacion_pacientes_sintomas_ADD_1['Sintoma3'])
Variable_predictoras['Box'] = le.fit_transform(df_relacion_pacientes_sintomas_ADD_1['Box'])
#Variable_predictoras.head(70)
if (Variable_predictoras == 'C2').any().any():
    print("'c1' is in the DataFrame.")
else:
    print("'c1' is not in the DataFrame.")

'c1' is not in the DataFrame.


#### Prediccion general

In [58]:
precision_global = np.sum(matriz_de_verificacion.diagonal()/np.sum(matriz_de_verificacion))
precision_global


0.33701657458563533

## Prediccion Box con Triage + Sintoma

#### Visualizacion matriz

In [59]:
df_relacion_pacientes_sintomas_ADD_2 = df_relacion_pacientes_sintomas.copy()
df_relacion_pacientes_sintomas_ADD_2.head()

,Paciente,Triage,Sintoma real,Sintoma1,Similitud1,Sintoma2,Similitud2,Sintoma3,Similitud3,Box
0,1,4.0,[fiebre tos],tos,0.707107,fiebre,0.707107,palpitaciones,0.0,18
1,2,4.0,[fiebre tos],tos,0.707107,fiebre,0.707107,palpitaciones,0.0,17
2,3,4.0,[tos],tos,1.000000,palpitaciones,0.000000,afasia,0.0,12
3,4,4.0,[hta],hta,1.000000,palpitaciones,0.000000,afasia,0.0,12
4,5,4.0,[fiebre],fiebre,1.000000,palpitaciones,0.000000,afasia,0.0,5


#### Preparo los datos

In [60]:
#df_relacion_pacientes_sintomas_ADD_2['Box'] = df_relacion_pacientes_sintomas_ADD_2['Box'].astype(str)

In [61]:
#En el tutorial de Arboles de decisión con Python es Y
Variable_a_predecir=df_relacion_pacientes_sintomas_ADD_2['Box']
# Crear el imputador
Variable_a_predecir.fillna(0, inplace=True)
# if Variable_a_predecir.isnull().values.any():
#     print("Hay valores nulos en el DataFrame. Reemplazándolos con 0...")
#     # Usamos la función fillna() para reemplazar los valores nulos con 0
#     Variable_a_predecir = Variable_a_predecir.fillna(0)
# else:
#     print("No hay valores nulos en el DataFrame.")
Variable_a_predecir.head()

0    18
1    17
2    12
3    12
4     5
Name: Box, dtype: object

In [62]:
#En el tutorial de Arboles de decisión con Python es X
Variable_predictoras=df_relacion_pacientes_sintomas_ADD_2.iloc[:,1:-1]
Variable_predictoras.drop('Sintoma real', axis=1, inplace=True)
Variable_predictoras.head()

,Triage,Sintoma1,Similitud1,Sintoma2,Similitud2,Sintoma3,Similitud3
0,4.0,tos,0.707107,fiebre,0.707107,palpitaciones,0.0
1,4.0,tos,0.707107,fiebre,0.707107,palpitaciones,0.0
2,4.0,tos,1.000000,palpitaciones,0.000000,afasia,0.0
3,4.0,hta,1.000000,palpitaciones,0.000000,afasia,0.0
4,4.0,fiebre,1.000000,palpitaciones,0.000000,afasia,0.0


#### Codifico los labeles de los sintomas

In [63]:
le = LabelEncoder()
Variable_predictoras['Sintoma1'] = le.fit_transform(df_relacion_pacientes_sintomas_ADD_2['Sintoma1'])
Variable_predictoras['Sintoma2'] = le.fit_transform(df_relacion_pacientes_sintomas_ADD_2['Sintoma2'])
Variable_predictoras['Sintoma3'] = le.fit_transform(df_relacion_pacientes_sintomas_ADD_2['Sintoma3'])
#Variable_predictoras.head(70)
Variable_predictoras.head()

,Triage,Sintoma1,Similitud1,Sintoma2,Similitud2,Sintoma3,Similitud3
0,4.0,15,0.707107,6,0.707107,5,0.0
1,4.0,15,0.707107,6,0.707107,5,0.0
2,4.0,15,1.000000,8,0.000000,0,0.0
3,4.0,11,1.000000,8,0.000000,0,0.0
4,4.0,10,1.000000,8,0.000000,0,0.0


#### Codifico los boxes (que fueron pasados a string) a un valor numerico

In [64]:
Variable_a_predecir.head()

0    18
1    17
2    12
3    12
4     5
Name: Box, dtype: object

In [65]:
Variable_a_predecir = Variable_a_predecir.apply(pd.to_numeric, errors='coerce')

# Get the maximum integer value in the DataFrame
max_int = Variable_a_predecir.max().max()
# Fill NaN values with max_int + 1
Variable_a_predecir = Variable_a_predecir.fillna(max_int + 1)
if (Variable_a_predecir == 'C1').any().any():
    print("'c1' is in the DataFrame.")
else:
    print("'c1' is not in the DataFrame.")
Variable_a_predecir.head()

'c1' is not in the DataFrame.


0    18.0
1    17.0
2    12.0
3    12.0
4     5.0
Name: Box, dtype: float64

#### Separo el data set en entrenamiento y en testing

In [66]:
x_train,x_test,y_train,y_test = train_test_split(Variable_predictoras,Variable_a_predecir,train_size=0.75,random_state=0)
x_train.info()
y_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 543 entries, 77 to 684
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Triage      482 non-null    float64
 1   Sintoma1    543 non-null    int32  
 2   Similitud1  543 non-null    float64
 3   Sintoma2    543 non-null    int32  
 4   Similitud2  543 non-null    float64
 5   Sintoma3    543 non-null    int32  
 6   Similitud3  543 non-null    float64
dtypes: float64(4), int32(3)
memory usage: 27.6 KB
<class 'pandas.core.series.Series'>
Index: 543 entries, 77 to 684
Series name: Box
Non-Null Count  Dtype  
--------------  -----  
543 non-null    float64
dtypes: float64(1)
memory usage: 8.5 KB


In [67]:
display(x_train.head(4))

,Triage,Sintoma1,Similitud1,Sintoma2,Similitud2,Sintoma3,Similitud3
77,2.0,11,1.000000,8,0.0,0,0.0
319,4.0,15,0.662168,8,0.0,0,0.0
261,4.0,10,0.529899,8,0.0,0,0.0
109,3.0,8,1.000000,8,0.0,0,0.0


In [68]:
for i in range(x_train.shape[1]):
    if isinstance(x_train.iloc[:, i].values[0], list):
        print(f"La característica {i} es una lista.")

In [69]:
#Creamos el modelo
arbol = DecisionTreeClassifier()

#Entrenamos el modelo
arbol_triage=arbol.fit(x_train,y_train)


In [70]:
Y_pred=arbol_triage.predict(x_test)
Y_pred

array([ 8., 18., 17.,  7.,  5., 17., 16., 14., 17., 14., 19., 17., 18.,
       19., 17.,  8.,  7.,  5., 14.,  7.,  5., 14., 19.,  8., 17., 12.,
       16.,  7.,  5.,  5., 17., 18., 19., 19., 17.,  6., 17., 18., 16.,
       18., 16.,  5.,  2.,  8., 16.,  8.,  5., 16., 17., 16., 16.,  8.,
       17., 12., 17., 17., 15., 16.,  8.,  6., 16., 12., 12.,  5., 18.,
        5.,  5., 17., 17., 17.,  5., 16.,  8., 16.,  8., 18.,  5.,  8.,
        5.,  1., 16.,  7.,  7., 16.,  9., 19., 16., 16., 18.,  8.,  5.,
       16., 19., 18., 19., 16., 14.,  5., 19., 17.,  7., 18., 16.,  7.,
        5.,  6.,  5., 16.,  5.,  1.,  7., 17.,  5., 16., 19.,  5.,  5.,
       14., 16., 17., 17., 18., 17., 16.,  5.,  5., 17., 18., 19., 14.,
        1., 16., 16.,  6., 16., 16., 16., 16.,  6., 16., 16.,  5.,  8.,
       14.,  8., 19., 19.,  8., 16., 19.,  5.,  8.,  5., 17., 17., 16.,
       16.,  5.,  6.,  2.,  7., 17., 16., 19., 17., 12., 12., 16., 14.,
       17.,  8., 16., 17.,  5.,  8., 19., 12., 18., 17., 17., 16

In [71]:
matriz_de_verificacion = confusion_matrix(y_test,Y_pred)
matriz_de_verificacion

array([[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 4, 0, 3, 3, 0, 0, 0, 0, 1, 0, 4, 2, 1, 0],
       [0, 0, 1, 0, 4, 0, 0, 0, 0, 0, 0, 0, 1, 0, 3, 3, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 1],
       [0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 2, 1, 0],
       [0, 0, 0, 0, 1, 0, 0, 2, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 2, 0, 0, 1, 0, 0, 0, 4, 0, 2, 0],
       [0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 4, 3, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 3, 0, 1],
       [0, 0, 0, 0, 4, 3, 1, 3, 0, 0, 0, 0, 1, 1, 7, 1, 3, 2],
       [0, 0, 0, 0, 1, 1, 2, 1, 0, 0, 0, 0, 1, 0, 3, 2,

In [72]:
precision_global = np.sum(matriz_de_verificacion.diagonal()/np.sum(matriz_de_verificacion))
precision_global


0.13812154696132595

## Decision Tree Triage Level bruto (Chequear si conviene borrarlo)

In [73]:

# df_relacion_pacientes_sintomas
# # Crea el codificador
# encoder = OneHotEncoder()

# # Ajusta el codificador y transforma las columnas de síntomas
# sintomas_encoded = encoder.fit_transform(df_relacion_pacientes_sintomas[['Sintoma1', 'Sintoma2', 'Sintoma3']])

# # Convierte la matriz dispersa en una matriz densa
# sintomas_encoded = sintomas_encoded.toarray()

# # Crea un DataFrame con las columnas codificadas
# feature_names = [f"x{str(i)}" for i in range(sintomas_encoded.shape[1])]
# df_sintomas_encoded = pd.DataFrame(sintomas_encoded, columns=feature_names)

# # Une el nuevo DataFrame con las columnas de similitud
# features = pd.concat([df_sintomas_encoded, df_relacion_pacientes_sintomas[['Similitud1', 'Similitud2', 'Similitud3']]], axis=1)

# # Supongamos que 'labels' es una serie o un DataFrame con las etiquetas de nivel de triage para cada paciente
# labels = df_relacion_pacientes_sintomas[['Triage']]
# # display(features.head(5))
# # display(labels.head(5))

# # Divide los datos en conjuntos de entrenamiento y prueba
# features_train, features_test, labels_train, labels_test = train_test_split(features, labels, test_size=0.2)

# # Reemplaza los valores no finitos con un número, por ejemplo, 0
# features_train = features_train.fillna(0)
# labels_train = labels_train.fillna(0)
# labels_test = labels_test.fillna(0)
# # Ahora puedes entrenar tu modelo
# tree = DecisionTreeClassifier()
# tree.fit(features_train, labels_train)

# # Evalúa el árbol de decisión
# labels_pred = tree.predict(features_test)
# print(classification_report(labels_test, labels_pred, zero_division=1))




## Decision tree Bruto BOX

In [78]:
df_relacion_pacientes_sintomas_ADD_3 = df_relacion_pacientes_sintomas.copy()

df_relacion_pacientes_sintomas_ADD_3.fillna(0, inplace=True)

df_relacion_pacientes_sintomas_ADD_3.drop('Sintoma real', axis=1, inplace=True)
df_relacion_pacientes_sintomas_ADD_3.drop('Paciente', axis=1, inplace=True)
df_relacion_pacientes_sintomas_ADD_3['Box'] = pd.to_numeric(df_relacion_pacientes_sintomas_ADD_3['Box'], errors='coerce')

# Eliminamos las filas con NaN en la columna 'Box'
df_relacion_pacientes_sintomas_ADD_3 = df_relacion_pacientes_sintomas_ADD_3.dropna(subset=['Box'])
df_relacion_pacientes_sintomas_ADD_3.info()
df_relacion_pacientes_sintomas_ADD_3.head()

<class 'pandas.core.frame.DataFrame'>
Index: 637 entries, 0 to 723
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Triage      637 non-null    float64
 1   Sintoma1    637 non-null    object 
 2   Similitud1  637 non-null    float64
 3   Sintoma2    637 non-null    object 
 4   Similitud2  637 non-null    float64
 5   Sintoma3    637 non-null    object 
 6   Similitud3  637 non-null    float64
 7   Box         637 non-null    float64
dtypes: float64(5), object(3)
memory usage: 44.8+ KB


,Triage,Sintoma1,Similitud1,Sintoma2,Similitud2,Sintoma3,Similitud3,Box
0,4.0,tos,0.707107,fiebre,0.707107,palpitaciones,0.0,18.0
1,4.0,tos,0.707107,fiebre,0.707107,palpitaciones,0.0,17.0
2,4.0,tos,1.000000,palpitaciones,0.000000,afasia,0.0,12.0
3,4.0,hta,1.000000,palpitaciones,0.000000,afasia,0.0,12.0
4,4.0,fiebre,1.000000,palpitaciones,0.000000,afasia,0.0,5.0


In [80]:
df_relacion_pacientes_sintomas_ADD_3 = df_relacion_pacientes_sintomas.copy()
# Crea el codificador
encoder = OneHotEncoder()

# Ajusta el codificador y transforma las columnas de síntomas
sintomas_encoded = encoder.fit_transform(df_relacion_pacientes_sintomas_ADD_3[['Sintoma1', 'Sintoma2', 'Sintoma3']])

# Convierte la matriz dispersa en una matriz densa
sintomas_encoded = sintomas_encoded.toarray()

# Crea un DataFrame con las columnas codificadas
feature_names = [f"x{str(i)}" for i in range(sintomas_encoded.shape[1])]
df_sintomas_encoded = pd.DataFrame(sintomas_encoded, columns=feature_names)

# Une el nuevo DataFrame con las columnas de similitud
features = pd.concat([df_sintomas_encoded, df_relacion_pacientes_sintomas_ADD_3[['Similitud1', 'Similitud2', 'Similitud3']]], axis=1)

# Ahora 'labels' es una serie o un DataFrame con las etiquetas de box para cada paciente
labels = df_relacion_pacientes_sintomas['Box'].astype(str)

# Divide los datos en conjuntos de entrenamiento y prueba
features_train, features_test, labels_train, labels_test = train_test_split(features, labels, test_size=0.2)

# Reemplaza los valores no finitos con un número, por ejemplo, 0
features_train = features_train.fillna(0)
labels_train = labels_train.fillna(0)
labels_test = labels_test.fillna(0)

# Ahora puedes entrenar tu modelo
tree = DecisionTreeClassifier()
tree.fit(features_train, labels_train)

# Evalúa el árbol de decisión
labels_pred = tree.predict(features_test)
print(classification_report(labels_test, labels_pred, zero_division=1))


              precision    recall  f1-score   support

           1       1.00      0.00      0.00         1
          11       1.00      0.00      0.00         1
          12       0.00      0.00      0.00         3
          13       0.00      0.00      0.00         4
          14       1.00      0.00      0.00         4
          15       0.00      0.00      0.00         8
          16       0.13      0.23      0.17        22
          17       0.19      0.27      0.22        22
          18       0.18      0.21      0.19        24
           3       1.00      0.00      0.00         1
           4       1.00      0.00      0.00         2
           5       0.00      0.00      0.00        11
           6       1.00      0.00      0.00         4
           7       1.00      0.00      0.00         9
           8       1.00      0.00      0.00         6
           9       1.00      0.00      0.00         5
          AF       1.00      0.00      0.00         1
           C       1.00    

## Random Forest clasifier

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Cargamos los datos
df_Random_Forest = df_relacion_pacientes_sintomas.copy()
df_Random_Forest.fillna(0, inplace=True)

# Hacemos One-Hot Encoding para las columnas 'Sintoma1', 'Sintoma2' y 'Sintoma3'
df_Random_Forest.drop('Sintoma real', axis=1, inplace=True)
df_Random_Forest.drop('Paciente', axis=1, inplace=True)
df_Random_Forest['Box'] = pd.to_numeric(df_Random_Forest['Box'], errors='coerce')

# Eliminamos las filas con NaN en la columna 'Box'
df_Random_Forest = df_Random_Forest.dropna(subset=['Box'])


df_Random_Forest = pd.get_dummies(df_Random_Forest, columns=['Sintoma1', 'Sintoma2', 'Sintoma3'])
df_Random_Forest.info()

<class 'pandas.core.frame.DataFrame'>
Index: 637 entries, 0 to 723
Data columns (total 39 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Triage                            637 non-null    float64
 1   Similitud1                        637 non-null    float64
 2   Similitud2                        637 non-null    float64
 3   Similitud3                        637 non-null    float64
 4   Box                               637 non-null    float64
 5   Sintoma1_afasia                   637 non-null    bool   
 6   Sintoma1_cefalea                  637 non-null    bool   
 7   Sintoma1_convulsiones             637 non-null    bool   
 8   Sintoma1_diarrea                  637 non-null    bool   
 9   Sintoma1_disnea                   637 non-null    bool   
 10  Sintoma1_dolor abdominal          637 non-null    bool   
 11  Sintoma1_dolor torácico           637 non-null    bool   
 12  Sintoma1_dors

In [ ]:

# Preprocesamos los datos
# Aquí puedes incluir el preprocesamiento necesario para tus datos
# Por ejemplo, podrías necesitar codificar las variables categóricas, normalizar las numéricas, etc.

# Dividimos los datos en conjuntos de entrenamiento y prueba
X = df_Random_Forest[['Triage', 'Similitud1', 'Similitud2', 'Similitud3']]
y = df_Random_Forest['Box']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenamos el modelo
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Hacemos las predicciones
y_pred = model.predict(X_test)

# Evaluamos el modelo
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')


Accuracy: 0.140625


## Decision Tree Clasifier 4.0


In [163]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

sintomas_rellenados_por_enfermeros = train_df['MOTIVO DE CONSULTA'].tolist()
resultados = []
sintomas_ideales = ['convulsiones', 'trauma de cráneo', 'dolor torácico', 'dorsal', 'dolor abdominal', 'lumbar', 'cefalea', 'déficit motor', 'disartria', 'afasia', 'pérdida aguda de visión', 'disnea', 'otro dolor en curso', 'sobredosis de fármacos', 'ingesta de tóxicos', 'sangrado digestivo', 'fiebre', 'tos','dt','hta','diarrea','palpitaciones']
sintomas_sin_similitud = {}

# Create a DataFrame to store the results
df_relacion_pacientes_sintomas_random_forest = pd.DataFrame(columns=['Paciente', 'Triage', 'Sintoma real', 'Box'] + sintomas_ideales)

for index, row in train_df.iterrows():
    sintoma_paciente = [row['MOTIVO DE CONSULTA']]
    textos = sintoma_paciente + sintomas_ideales
    vectorizer = TfidfVectorizer().fit_transform(textos)
    similitud_coseno = cosine_similarity(vectorizer)
    indices_ordenados = np.argsort(similitud_coseno[0][1:])[::-1]

    if similitud_coseno[0][indices_ordenados[0]+1] == 0:
        if sintoma_paciente[0] in sintomas_sin_similitud:
            sintomas_sin_similitud[sintoma_paciente[0]] += 1
        else:
            sintomas_sin_similitud[sintoma_paciente[0]] = 1
    else:
        # Create a dictionary to store the patient's data
        paciente = {
            'Triage': train_df['TRIAGE'][index],
            'Box': train_df['BOX'][index]
        }
        # Add the similarity values for each ideal symptom
        for i in range(len(sintomas_ideales)):
            paciente[sintomas_ideales[i]] = similitud_coseno[0][i+1]
        # Append the patient's data to the results list
        resultados.append(paciente)

df_relacion_pacientes_sintomas_random_forest = pd.DataFrame(resultados)

df_relacion_pacientes_sintomas_random_forest = df_relacion_pacientes_sintomas_random_forest[df_relacion_pacientes_sintomas_random_forest['Triage'].notna()]
df_relacion_pacientes_sintomas_random_forest['Box'] = pd.to_numeric(df_relacion_pacientes_sintomas_random_forest['Box'], errors='coerce').fillna(0)
df_relacion_pacientes_sintomas_random_forest=df_relacion_pacientes_sintomas_random_forest.dropna()

display(df_relacion_pacientes_sintomas_random_forest.head())
if (df_relacion_pacientes_sintomas_random_forest == 'C2').any().any():
    print("'c1' is in the DataFrame.")
else:
    print("'c1' is not in the DataFrame.")

sintomas_sin_similitud_ordenados = {k: v for k, v in sorted(sintomas_sin_similitud.items(), key=lambda item: item[1], reverse=True)}
cantidad_de_sintmas=sum(sintomas_sin_similitud.values())
display("Cantidad de sintomas sin alguna similitud exitosa: "+str(cantidad_de_sintmas) )


,Triage,Box,convulsiones,trauma de cráneo,dolor torácico,dorsal,dolor abdominal,lumbar,cefalea,déficit motor,disartria,afasia,pérdida aguda de visión,disnea,otro dolor en curso,sobredosis de fármacos,ingesta de tóxicos,sangrado digestivo,fiebre,tos,dt,hta,diarrea,palpitaciones
0,4.0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,0.0
1,4.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,0.0
2,4.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0,0.0
3,4.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1.0,0.0,0.0
4,4.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,0.000000,0.0,0.0,0.0,0.0


'c1' is not in the DataFrame.


'Cantidad de sintomas sin alguna similitud exitosa: 701'

In [164]:
df_relacion_pacientes_sintomas_random_forest.info()

<class 'pandas.core.frame.DataFrame'>
Index: 648 entries, 0 to 719
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Triage                   648 non-null    float64
 1   Box                      648 non-null    float64
 2   convulsiones             648 non-null    float64
 3   trauma de cráneo         648 non-null    float64
 4   dolor torácico           648 non-null    float64
 5   dorsal                   648 non-null    float64
 6   dolor abdominal          648 non-null    float64
 7   lumbar                   648 non-null    float64
 8   cefalea                  648 non-null    float64
 9   déficit motor            648 non-null    float64
 10  disartria                648 non-null    float64
 11  afasia                   648 non-null    float64
 12  pérdida aguda de visión  648 non-null    float64
 13  disnea                   648 non-null    float64
 14  otro dolor en curso      648 no

In [165]:
df_Random_Forest2 = df_relacion_pacientes_sintomas_random_forest.copy()
df_Random_Forest2.head()

,Triage,Box,convulsiones,trauma de cráneo,dolor torácico,dorsal,dolor abdominal,lumbar,cefalea,déficit motor,disartria,afasia,pérdida aguda de visión,disnea,otro dolor en curso,sobredosis de fármacos,ingesta de tóxicos,sangrado digestivo,fiebre,tos,dt,hta,diarrea,palpitaciones
0,4.0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,0.0
1,4.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,0.0
2,4.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0,0.0
3,4.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1.0,0.0,0.0
4,4.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,0.000000,0.0,0.0,0.0,0.0


In [166]:
for i in range(df_Random_Forest2.shape[1]):
    if isinstance(df_Random_Forest2.iloc[:, i].values[0], list):
        print(f"La característica {i} es una lista. Los valores son: {df_Random_Forest2.iloc[:, i].values[0]}")
        # Encuentra los índices de las filas que contienen listas
        indices = [idx for idx, val in enumerate(df_Random_Forest2.iloc[:, i]) if isinstance(val, list)]
        # Elimina estas filas
        df_Random_Forest2 = df_Random_Forest2.drop(df_Random_Forest2.index[indices])


In [168]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# Separar las características (X) de la etiqueta objetivo (y)
X_randomforest = df_Random_Forest2.drop('Box', axis=1)
y_randomforest = df_Random_Forest2['Box']

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_randomforest, y_randomforest, test_size=0.8, random_state=0)

# Crear el modelo de árbol de decisión
clf = DecisionTreeClassifier()

# Entrenar el modelo
clf.fit(X_train, y_train)

# Hacer predicciones en el conjunto de prueba
y_pred = clf.predict(X_test)

# Calcular la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)

print(f'La precisión del modelo de árbol de decisión es: {accuracy}')


La precisión del modelo de árbol de decisión es: 0.15606936416184972


## Random Forest 3.0

In [ ]:
df_Random_Forest3 = df_relacion_pacientes_sintomas.copy()


# Hacemos One-Hot Encoding para las columnas 'Sintoma1', 'Sintoma2' y 'Sintoma3'
df_Random_Forest3.drop('Sintoma real', axis=1, inplace=True)
df_Random_Forest3.drop('Paciente', axis=1, inplace=True)
df_Random_Forest3['Box'] = pd.to_numeric(df_Random_Forest3['Box'], errors='coerce')

# Eliminamos las filas con NaN en la columna 'Box'
df_Random_Forest3 = df_Random_Forest3.dropna(subset=['Box'])

In [90]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score


# Supongamos que 'df' es tu DataFrame
le = LabelEncoder()

# Codificar las variables categóricas
df_Random_Forest3['Sintoma1'] = le.fit_transform(df_Random_Forest3['Sintoma1'])
df_Random_Forest3['Sintoma2'] = le.fit_transform(df_Random_Forest3['Sintoma2'])
df_Random_Forest3['Sintoma3'] = le.fit_transform(df_Random_Forest3['Sintoma3'])

# Definir las variables independientes y la variable objetivo
X = df_Random_Forest3[['Triage', 'Sintoma1', 'Similitud1', 'Sintoma2', 'Similitud2', 'Sintoma3', 'Similitud3']]
y = df_Random_Forest3['Box']

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear y entrenar el modelo
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# Hacer predicciones con el conjunto de prueba
predictions = model.predict(X_test)
# Calcular la precisión
accuracy = accuracy_score(y_test, predictions)

print(f'La precisión del modelo es: {accuracy}')

La precisión del modelo es: 0.1328125


## Red Neuronal

In [93]:
df_Red_Neuronal = df_relacion_pacientes_sintomas.copy()
df_Red_Neuronal.fillna(0, inplace=True)

# Hacemos One-Hot Encoding para las columnas 'Sintoma1', 'Sintoma2' y 'Sintoma3'
df_Red_Neuronal.drop('Sintoma real', axis=1, inplace=True)
df_Red_Neuronal.drop('Paciente', axis=1, inplace=True)
df_Red_Neuronal['Box'] = pd.to_numeric(df_Red_Neuronal['Box'], errors='coerce')

# Eliminamos las filas con NaN en la columna 'Box'
df_Red_Neuronal = df_Red_Neuronal.dropna(subset=['Box'])

In [95]:
df_Red_Neuronal.head(20)

,Triage,Sintoma1,Similitud1,Sintoma2,Similitud2,Sintoma3,Similitud3,Box
0,4.0,tos,0.707107,fiebre,0.707107,palpitaciones,0.000000,18.0
1,4.0,tos,0.707107,fiebre,0.707107,palpitaciones,0.000000,17.0
2,4.0,tos,1.000000,palpitaciones,0.000000,afasia,0.000000,12.0
3,4.0,hta,1.000000,palpitaciones,0.000000,afasia,0.000000,12.0
4,4.0,fiebre,1.000000,palpitaciones,0.000000,afasia,0.000000,5.0
5,3.0,sangrado digestivo,0.350882,palpitaciones,0.000000,afasia,0.000000,16.0
6,4.0,fiebre,1.000000,palpitaciones,0.000000,afasia,0.000000,5.0
7,4.0,dolor torácico,0.274220,dolor abdominal,0.274220,otro dolor en curso,0.180975,16.0
8,2.0,lumbar,0.767925,dolor torácico,0.380043,dolor abdominal,0.380043,18.0
9,4.0,dolor torácico,0.232323,dolor abdominal,0.232323,otro dolor en curso,0.153325,17.0


In [94]:
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

# Preprocesamiento de los datos
encoder = LabelEncoder()
sintoma1 = encoder.fit_transform(df_Red_Neuronal['Sintoma1'])
sintoma2 = encoder.fit_transform(df_Red_Neuronal['Sintoma2'])
sintoma3 = encoder.fit_transform(df_Red_Neuronal['Sintoma3'])

X = pd.DataFrame({
    'Triage': df_Red_Neuronal['Triage'],
    'Sintoma1': sintoma1,
    'Similitud1': df_Red_Neuronal['Similitud1'],
    'Sintoma2': sintoma2,
    'Similitud2': df_Red_Neuronal['Similitud2'],
    'Sintoma3': sintoma3,
    'Similitud3': df_Red_Neuronal['Similitud3']
})

y = to_categorical(df_Red_Neuronal['Box'])

# Creación del modelo
model = Sequential()
model.add(Dense(32, input_dim=7, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(y.shape[1], activation='softmax'))  # La cantidad de neuronas en la capa de salida debe ser igual al número de clases

# Compilación del modelo
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Entrenamiento del modelo
model.fit(X, y, epochs=50, batch_size=10)


Epoch 1/50


C:\Users\Adri\AppData\Roaming\Python\Python310\site-packages\keras\src\layers\core\dense.py:85: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 806us/step - accuracy: 0.0771 - loss: 3.8212 
Epoch 2/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 645us/step - accuracy: 0.0776 - loss: 2.7571
Epoch 3/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 620us/step - accuracy: 0.1388 - loss: 2.6328
Epoch 4/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 690us/step - accuracy: 0.1358 - loss: 2.6354   
Epoch 5/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 663us/step - accuracy: 0.1413 - loss: 2.5770   
Epoch 6/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1703 - loss: 2.5545   
Epoch 7/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 653us/step - accuracy: 0.1709 - loss: 2.5181
Epoch 8/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 646us/step - accuracy: 0.1834 - loss: 2.5186
Epoch 9/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 651us/step - accuracy: 0.1645 - loss: 2.5374
Epoch 10/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 652us/step - accuracy: 0.1658 - loss: 2.5572
Epoch 11/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 707us/step - accuracy: 0.1717 - loss: 2.5666
Epoch 12/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 656us/step - 

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

# Lista de síntomas
sintomas = [
    'Convulsiones',
    'Trauma de Cráneo',
    'Dolor torácico / dorsal',
    'Dolor abdominal / lumbar',
    'Cefalea',
    'Déficit motor',
    'Disartria - afasia',
    'Pérdida aguda de visión',
    'Disnea',
    'Otro dolor en curso',
    'Sobredosis de fármacos / Ingesta de tóxicos',
    'Sangrado Digestivo',
    'Fiebre >38°'
]

# Asumiendo que 'model' es un modelo Word2Vec entrenado
vectores_palabras = [model.wv[sintoma] for sintoma in sintomas if sintoma in model.wv]

# Verificar si la lista de vectores de palabras no está vacía
if vectores_palabras:
    X = np.array(vectores_palabras)

    # El número de clusters debe ser menor o igual al número de síntomas
    kmeans = KMeans(n_clusters=min(13, len(sintomas)))
    kmeans.fit(X)

    etiquetas_clusters = kmeans.labels_
    clusters_palabras = {sintoma: etiqueta for sintoma, etiqueta in zip(sintomas, etiquetas_clusters)}

    for sintoma, etiqueta in clusters_palabras.items():
        print(f"{sintoma}: Clúster {etiqueta}")
else:
    print("Ninguno de los síntomas está en el vocabulario del modelo Word2Vec.")


NameError: name 'model' is not defined

In [ ]:
#Usar pca para poder graficar y visualizar en dos dimensiones

# Use PCA to reduce the dimensionality of the word vectors to 2D
pca = PCA(n_components=2)
word_vectors_2d = pca.fit_transform(word_vectors)
# Create a DataFrame with the 2D word vectors and their corresponding words and cluster labels
df = pd.DataFrame(word_vectors_2d, columns=['Component 1', 'Component 2'])
df['Word'] = model.wv.index_to_key
df['Cluster Label'] = cluster_labels

# Create a scatter plot of the 2D word vectors, colored by their cluster labels, and with hover text for the words
fig = px.scatter(df, x='Component 1', y='Component 2', color='Cluster Label', hover_data=['Word'])
fig.show()

In [ ]:
# Use PCA to reduce the dimensionality of the word vectors to 3D
pca = PCA(n_components=3)
word_vectors_3d = pca.fit_transform(word_vectors)

# Create a DataFrame with the 3D word vectors and their corresponding words and cluster labels
df = pd.DataFrame(word_vectors_3d, columns=['Component 1', 'Component 2', 'Component 3'])
df['Word'] = model.wv.index_to_key
df['Cluster Label'] = cluster_labels

# Create a 3D scatter plot of the word vectors, colored by their cluster labels, and with hover text for the words
fig = go.Figure(data=[go.Scatter3d(
    x=df['Component 1'],
    y=df['Component 2'],
    z=df['Component 3'],
    mode='markers',
    marker=dict(
        size=12,
        color=df['Cluster Label'],                # set color to an array/list of desired values
        colorscale='Viridis',   # choose a colorscale
        opacity=0.8
    ),
    text=df['Word']
)])

# tight layout
fig.update_layout(margin=dict(l=0, r=0, b=0, t=0))
fig.show()